# Titanic - Machine Learning from Disaster

**Objective:** Use machine learning to create a model that predicts which passengers survived the Titanic shipwreck.

In [21]:
from datasets import load_dataset
import pandas as pd

# Load the dataset from Hugging Face
dataset = load_dataset("amgadmadkour/titanic")
train_data = dataset["train"]
test_data = dataset["test"]

# Convert to pandas DataFrame
train_data = train_data.to_pandas()
test_data = test_data.to_pandas()

In [22]:
len(test_data)

418

In [23]:
women = train_data.loc[train_data.Sex == 'female']["Survived"]
rate_women = sum(women)/len(women)

print("% of women who survived:", rate_women)

% of women who survived: 0.7420382165605095


In [24]:
men = train_data.loc[train_data.Sex == 'male']["Survived"]
rate_men = sum(men)/len(men)

print("% of men who survived:", rate_men)

% of men who survived: 0.18890814558058924


In [25]:
test_data.columns

Index(['PassengerId', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp', 'Parch',
       'Ticket', 'Fare', 'Cabin', 'Embarked', 'Survived'],
      dtype='str')

In [26]:
test_data.index

RangeIndex(start=0, stop=418, step=1)

In [27]:
features = train_data.columns.drop('Survived').tolist()
features

['PassengerId',
 'Pclass',
 'Name',
 'Sex',
 'Age',
 'SibSp',
 'Parch',
 'Ticket',
 'Fare',
 'Cabin',
 'Embarked']

In [37]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, VotingClassifier, StackingClassifier
from sklearn.metrics import accuracy_score

features = ["Pclass", "Sex", "Age", "SibSp", "Parch", "Fare", "Embarked"]

X = train_data[features].copy()

X["Age"] = X["Age"].fillna(X["Age"].median())
X["Fare"] = X["Fare"].fillna(X["Fare"].median())
X["Embarked"] = X["Embarked"].fillna(X["Embarked"].mode()[0])

X = pd.get_dummies(X, drop_first=True)
y = train_data["Survived"]

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=1, stratify=y
)

ensemble_model = VotingClassifier(
    estimators=[
        ("lr", LogisticRegression(max_iter=1000)),
        ("rf", RandomForestClassifier(n_estimators=100, max_depth=5, random_state=1)),
        ("gb", GradientBoostingClassifier(random_state=1)),
    ],
    voting="soft"
)

ensemble_model.fit(X_train, y_train)

val_preds = ensemble_model.predict(X_val)
val_accuracy = accuracy_score(y_val, val_preds)

print("Validation accuracy:", val_accuracy)

Validation accuracy: 0.8268156424581006


In [38]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Random Forest": RandomForestClassifier(n_estimators=100, max_depth=5, random_state=1),
    "Gradient Boosting": GradientBoostingClassifier(random_state=1),
    "Stacking": StackingClassifier(
        estimators=[
            ("lr", LogisticRegression(max_iter=1000)),
            ("rf", RandomForestClassifier(n_estimators=100, max_depth=5, random_state=1)),
            ("gb", GradientBoostingClassifier(random_state=1)),
        ],
        final_estimator=LogisticRegression(max_iter=1000),
        cv=5
    )
}

for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_val)
    acc = accuracy_score(y_val, preds)
    print(f"{name}: {acc:.4f}")

Logistic Regression: 0.7821
Random Forest: 0.8492
Gradient Boosting: 0.8324
Stacking: 0.8380


In [43]:
X_test = pd.get_dummies(test_data[features])
X_test = X_test.reindex(columns=X.columns, fill_value=0)

predictions = models["Random Forest"].predict(X_test)
output = pd.DataFrame({'PassengerId': test_data.PassengerId, 'Survived': predictions})
output.to_csv('submission.csv', index=False)
print("Submission was successfully saved!")

Submission was successfully saved!
